# EDA: جداول `orders` و `order_payments` (Olist Brazil E-commerce)

می‌خواهیم قبل از هر تحلیل جدی بفهمیم این دو جدول دقیقاً چه چیزی را نشان می‌دهند: هر ردیف یعنی چه، چه چیزی در آن‌ها ناقص یا عجیب است و کجا باید مراقب بود. نمودار نمی‌کشیم و فقط عدد و جدول می‌بینیم.
هر بخش با چند خط توضیح شروع می‌شود که می‌گوید چه چیزی را چک می‌کنیم و چرا. جمع‌بندی کامل هم در `docs/eda_orders_payments.md` آمده است.

**فایل‌ها:** `olist_orders_dataset.csv` و `olist_order_payments_dataset.csv` را در مسیر `data/raw/olist/` قرار دارند.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50, 'display.width', 200)

# مسیر داده: نسبت به پوشه notebooks/01_EDA
RAW = Path('../../data/raw/olist')
orders = pd.read_csv(RAW / 'olist_orders_dataset.csv')
pay = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
print('orders:', orders.shape, '| payments:', pay.shape)

## ۱. جدول `orders`
هر ردیف **یک سفارش** است (کلید: `order_id`) و وضعیت سفارش و زمان‌بندی مراحل آن (خرید، تأیید پرداخت، تحویل به پست، تحویل به مشتری، تاریخ تخمینی) را نگه می‌دارد.

### ۱.۱ ساختار، نوع ستون‌ها و نمونه

In [ ]:
display(orders.head())
orders.info()

### ۱.۲ مقادیر null، ردیف تکراری و یکتایی کلیدها
ستون‌های تاریخ به صورت رشته خوانده شده‌اند و باید به `datetime` تبدیل شوند.
null در ستون‌های تاریخ لزوماً خطا نیست و ممکن است فقط یعنی سفارش هنوز به آن مرحله نرسیده است. این را در بخش بعد با `order_status` مقایسه می‌کنیم.

In [ ]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

null_tbl = pd.DataFrame({'null': orders.isna().sum(), 'null_%': (orders.isna().mean()*100).round(2)})
display(null_tbl)
print('ردیف کاملاً تکراری:', orders.duplicated().sum())
print('order_id تکراری :', orders.order_id.duplicated().sum())
print('customer_id تکراری:', orders.customer_id.duplicated().sum(), '(در Olist هر سفارش customer_id مخصوص خودش را دارد؛ برای شناخت مشتری تکراری باید به customer_unique_id در جدول customers رفت)')

### ۱.۳ توزیع وضعیت سفارش‌ها

In [ ]:
display(orders.order_status.value_counts().to_frame('count').assign(pct=lambda d: (d['count']/len(orders)*100).round(2)))

### ۱.۴ ارتباط null تاریخ‌ها با وضعیت سفارش
اگر null ها با وضعیت سازگار باشند (مثلاً `shipped` تاریخ تحویل به مشتری ندارد) طبیعی‌اند. موارد ناسازگار (مثل `delivered` بدون تاریخ تحویل) ناهنجاری داده هستند.

In [ ]:
display(orders.groupby('order_status')[date_cols].apply(lambda d: d.isna().sum()))

print('delivered ولی بدون تاریخ تحویل به مشتری:', ((orders.order_status=='delivered') & orders.order_delivered_customer_date.isna()).sum())
print('غیر delivered ولی دارای تاریخ تحویل به مشتری:', ((orders.order_status!='delivered') & orders.order_delivered_customer_date.notna()).sum())

### ۱.۵ منطقی بودن ترتیب زمانی
ترتیب انتظار: خرید ← تأیید ← تحویل به پست ← تحویل به مشتری. هر مورد خلاف ترتیب یک ناهنجاری احتمالی است (خطای ثبت یا ثبت دستی).

In [ ]:
checks = {
 'approved < purchase'          : orders.order_approved_at < orders.order_purchase_timestamp,
 'carrier < approved'           : orders.order_delivered_carrier_date < orders.order_approved_at,
 'customer < carrier'           : orders.order_delivered_customer_date < orders.order_delivered_carrier_date,
 'customer < purchase'          : orders.order_delivered_customer_date < orders.order_purchase_timestamp,
 'carrier < purchase'           : orders.order_delivered_carrier_date < orders.order_purchase_timestamp,
}
display(pd.Series({k: int(v.sum()) for k, v in checks.items()}, name='تعداد ردیف ناسازگار'))
print('بازه تاریخ خرید:', orders.order_purchase_timestamp.min(), '→', orders.order_purchase_timestamp.max())

### ۱.۶ توزیع و داده‌های پرت در مدت‌زمان‌ها
از تاریخ‌ها چند متغیر عددی (روز) می‌سازیم: زمان تأیید، زمان تحویل کل، مدت تخمینی و میزان تأخیر نسبت به تخمین. سپس پرت‌ها را با IQR بررسی می‌کنیم.

In [ ]:
days = lambda a, b: (a - b).dt.total_seconds() / 86400
dur = pd.DataFrame({
  'approve_days'   : days(orders.order_approved_at, orders.order_purchase_timestamp),
  'delivery_days'  : days(orders.order_delivered_customer_date, orders.order_purchase_timestamp),
  'estimated_days' : days(orders.order_estimated_delivery_date, orders.order_purchase_timestamp),
  'delay_vs_est'   : days(orders.order_delivered_customer_date, orders.order_estimated_delivery_date),
})
display(dur.describe(percentiles=[.01,.25,.5,.75,.95,.99,.999]).round(2))
print('skewness:'); display(dur.skew().round(2))

def iqr_outliers(s, k=1.5):
    s = s.dropna(); q1, q3 = s.quantile([.25,.75]); i = q3-q1
    lo, hi = q1-k*i, q3+k*i
    return pd.Series({'lower':lo,'upper':hi,'n_out':int(((s<lo)|(s>hi)).sum()),'out_%':round(((s<lo)|(s>hi)).mean()*100,2)})
display(dur.apply(iqr_outliers).T)

delivered = orders.dropna(subset=['order_delivered_customer_date'])
print('تحویل دیرتر از تاریخ تخمینی:', (delivered.order_delivered_customer_date > delivered.order_estimated_delivery_date).sum(),
      f"({(delivered.order_delivered_customer_date > delivered.order_estimated_delivery_date).mean()*100:.1f}% از سفارش‌های تحویل‌شده)")

### ۱.۷ توزیع سفارش‌ها در طول زمان (ماهانه)

In [ ]:
monthly = orders.groupby(orders.order_purchase_timestamp.dt.to_period('M')).size().rename('orders')
display(monthly.to_frame().T)

### ۱.۸ الگوی روز هفته و ساعت خرید
این بخش به ما می‌گوید مشتری‌ها چه موقع سفارش می‌دهند. ساعت‌ها بر اساس خود داده هستند و منطقه زمانی آن‌ها معلوم نیست، پس عدد دقیق ساعت را زیاد جدی نگیرید و فقط الگو را ببینید.

In [ ]:
t = orders.order_purchase_timestamp
display(t.dt.day_name().value_counts().to_frame('orders').T)
display(t.dt.hour.value_counts().sort_index().to_frame('orders').T)

### ۱.۹ نرخ تأخیر در تحویل به تفکیک ماه
تأخیر یعنی تحویل دیرتر از تاریخ تخمینی. در این مرحله فقط به همین سفارش‌ها نگاه می‌کنیم که تاریخ تحویل دارند. ماه‌هایی که سفارش بسیار کم دارند (مثل ۲۰۱۶) عدد پایداری نمی‌دهند.

In [ ]:
dl = orders[orders.order_status=='delivered'].dropna(subset=['order_delivered_customer_date']).copy()
dl['late'] = dl.order_delivered_customer_date > dl.order_estimated_delivery_date
lm = dl.groupby(dl.order_purchase_timestamp.dt.to_period('M')).agg(n=('late','size'), late_rate=('late','mean')).round(3)
display(lm.T)

## ۲. جدول `order_payments`
هر ردیف **یک تراکنش پرداخت** برای یک سفارش است. یک سفارش می‌تواند چند ردیف داشته باشد (مثلاً بخشی با voucher و بقیه با کارت) و `payment_sequential` شماره ترتیب آن‌هاست. کلید یکتا ترکیب (`order_id`, `payment_sequential`) است.

### ۲.۱ ساختار، null و تکراری

In [ ]:
display(pay.head())
pay.info()
display(pd.DataFrame({'null': pay.isna().sum(), 'nunique': pay.nunique()}))
print('ردیف کاملاً تکراری:', pay.duplicated().sum())
print('تکراری روی (order_id, payment_sequential):', pay.duplicated(['order_id','payment_sequential']).sum())

### ۲.۲ توزیع روش پرداخت

In [ ]:
display(pay.payment_type.value_counts().to_frame('count').assign(pct=lambda d: (d['count']/len(pay)*100).round(2)))
display(pay.describe().round(2))

### ۲.۳ مقادیر مشکوک
- `payment_value == 0`
- `payment_installments == 0`
- `payment_type == 'not_defined'`
- اقساط روی روش‌هایی غیر از کارت اعتباری (انتظار: فقط credit_card قسط دارد)

In [ ]:
print('payment_value = 0   :', (pay.payment_value==0).sum()); display(pay[pay.payment_value==0].payment_type.value_counts())
print('installments = 0    :', (pay.payment_installments==0).sum()); display(pay[pay.payment_installments==0])
print('not_defined         :', (pay.payment_type=='not_defined').sum())
print('قسط>1 در غیر کارت اعتباری:', ((pay.payment_type!='credit_card') & (pay.payment_installments>1)).sum())
display(pay.groupby('payment_type').payment_installments.describe())

### ۲.۴ توزیع و داده‌های پرت `payment_value`
توزیع مبلغ بسیار **راست‌چوله** است. بنابراین هم IQR و هم Z-score را می‌گیریم و درصدک‌های بالا را هم می‌بینیم.

In [ ]:
v = pay.payment_value
print(v.describe(percentiles=[.5,.9,.95,.99,.999]).round(2))
print('skewness:', round(v.skew(),2), '| kurtosis:', round(v.kurt(),2))
q1, q3 = v.quantile([.25,.75]); iqr = q3-q1
print('IQR upper fence:', round(q3+1.5*iqr,2), '| پرت‌ها:', (v>q3+1.5*iqr).sum(), f"({(v>q3+1.5*iqr).mean()*100:.1f}%)")
z = (v - v.mean())/v.std()
print('Z>3:', (z>3).sum())
print('log1p skew:', round(np.log1p(v).skew(),2))
display(pay.groupby('payment_type').payment_value.describe().round(2))
display(pay.nlargest(10,'payment_value'))

### ۲.۵ تعداد پرداخت به ازای هر سفارش

In [ ]:
per_order = pay.groupby('order_id').agg(n_pay=('payment_sequential','size'),
                                        n_types=('payment_type','nunique'),
                                        total=('payment_value','sum'))
display(per_order.n_pay.value_counts().sort_index().head(10).to_frame('orders'))
print('سفارش با بیش از یک پرداخت:', (per_order.n_pay>1).sum())
print('سفارش با بیش از یک روش پرداخت:', (per_order.n_types>1).sum())
print('max payment_sequential:', pay.payment_sequential.max())
display(per_order.total.describe().round(2))

### ۲.۶ ترکیب روش‌های پرداخت در یک سفارش
بیشتر سفارش‌ها با یک روش پرداخت می‌شوند. ببینیم ترکیب‌های چندروشی کدام‌اند.

In [ ]:
combo = pay.groupby('order_id').payment_type.apply(lambda s: '+'.join(sorted(set(s))))
display(combo.value_counts().to_frame('orders'))

### ۲.۷ رابطه تعداد اقساط و مبلغ (فقط کارت اعتباری)
انتظار داریم خریدهای گران‌تر در اقساط بیشتری پرداخت شوند. همبستگی اسپیرمن را هم حساب می‌کنیم چون توزیع مبلغ چوله است.

In [ ]:
cc = pay[pay.payment_type=='credit_card']
bins = pd.cut(cc.payment_installments, [-1,1,3,6,10,24], labels=['0-1','2-3','4-6','7-10','11-24'])
display(cc.groupby(bins, observed=True).payment_value.agg(['count','median','mean']).round(1))
print('Spearman(installments, value):', round(cc[['payment_installments','payment_value']].corr(method='spearman').iloc[0,1], 2))

### ۲.۸ مبلغ پرداخت در طول زمان
مجموع مبلغ پرداخت‌ها به تفکیک فصل (فقط سفارش‌های `delivered`). این فقط برای فهمیدن مقیاس و رشد داده است و گزارش مالی نیست.

In [ ]:
mm = pay.merge(orders[['order_id','order_status','order_purchase_timestamp']], on='order_id')
dd = mm[mm.order_status=='delivered']
display(dd.groupby(dd.order_purchase_timestamp.dt.to_period('Q')).payment_value.agg(['count','sum']).round(0))
print('مجموع کل پرداخت‌ها:', round(pay.payment_value.sum()), '| فقط delivered:', round(dd.payment_value.sum()))

## ۳. ارتباط دو جدول
رابطه: یک سفارش ← چند پرداخت (1:N) روی `order_id`.

In [ ]:
print('سفارش بدون هیچ پرداخت:', (~orders.order_id.isin(pay.order_id)).sum())
display(orders[~orders.order_id.isin(pay.order_id)])
print('پرداخت بدون سفارش (یتیم):', (~pay.order_id.isin(orders.order_id)).sum())
m = pay.merge(orders[['order_id','order_status']], on='order_id', how='left')
display(m.groupby('order_status').agg(rows=('order_id','size'), orders=('order_id','nunique'), total=('payment_value','sum')).round(2))
print('سفارش‌های not_defined:'); display(orders[orders.order_id.isin(pay[pay.payment_type=='not_defined'].order_id)][['order_id','order_status']])

## ۴. جمع‌بندی و نکات برای مراحل بعد
جمع‌بندی کامل با اعداد و توصیه‌ها در `docs/eda_orders_payments.md` آمده است.